# 09 - Reranker Fine-Tuning

**Pipeline position:** Step 9 of the ImmunoBiology RAG pipeline (after training-data generation in Notebook 08, before LLM SFT in Notebook 10).

## Purpose

Fine-tune the **BAAI/bge-reranker-v2-m3** cross-encoder on immunology-specific query-passage triplets so that the reranker learns domain-specific relevance signals (e.g., cytokine names, MHC subtypes, signaling pathways) that the generic model misses.

## Learning Objectives

1. Understand how a cross-encoder reranker differs from a bi-encoder retriever.
2. Load and inspect the triplet training data (`query / pos / neg`).
3. Configure training hyperparameters (learning rate, warmup, batch size).
4. Launch fine-tuning with periodic evaluation (NDCG@10, MRR@10).
5. Visualize training curves and compare pre-trained vs. fine-tuned performance.

## Key Inputs / Outputs

| Direction | File | Description |
|-----------|------|-------------|
| Input | `data/train/reranker_train.jsonl` | Training triplets |
| Input | `data/train/reranker_test.jsonl` | Evaluation triplets |
| Output | `outputs/models/reranker_finetuned/best/` | Best checkpoint |
| Output | `outputs/reranker_eval/*.png` | Training curves and comparison chart |

## Cell 2 -- Imports

Load all dependencies upfront so that missing packages surface immediately.

In [ ]:
import json
import sys
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt

# Ensure the project root is on sys.path so that `src` and `train` are importable
PROJECT_ROOT = Path.cwd().parent  # assumes notebooks/ is one level below root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import constant
from train.train_reranker import (
    RerankerDataset,
    train_reranker,
    plot_training_curves,
    compare_pre_post,
)

print(f"Project root : {PROJECT_ROOT}")
print(f"Train file   : {constant.reranker_train_path}")
print(f"Eval file    : {constant.reranker_test_path}")

## Cell 3 -- Load Training Data and Show Statistics

Before training we inspect the data to verify format correctness and understand the distribution of query/passage lengths. Each JSONL record has the schema `{"query": str, "pos": str, "neg": str}`.

In [ ]:
# ------------------------------------------------------------------
# Load raw JSONL and compute descriptive statistics
# ------------------------------------------------------------------
def load_jsonl(path: str):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

train_data = load_jsonl(constant.reranker_train_path)
eval_data  = load_jsonl(constant.reranker_test_path)

print(f"Training triplets : {len(train_data)}")
print(f"Evaluation triplets: {len(eval_data)}")
print(f"\nSample record (train):")
print(json.dumps(train_data[0], indent=2, ensure_ascii=False)[:600])

# Length distributions
query_lens = [len(r["query"].split()) for r in train_data]
pos_lens   = [len(r["pos"].split())   for r in train_data]
neg_lens   = [len(r["neg"].split())   for r in train_data]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, data, title, color in zip(
    axes,
    [query_lens, pos_lens, neg_lens],
    ["Query length (words)", "Positive passage length", "Negative passage length"],
    ["steelblue", "mediumseagreen", "tomato"],
):
    ax.hist(data, bins=40, color=color, alpha=0.8, edgecolor="white")
    ax.set_title(title)
    ax.set_xlabel("Word count")
    ax.set_ylabel("Frequency")
    ax.axvline(np.mean(data), color="black", linestyle="--", linewidth=1,
               label=f"mean={np.mean(data):.0f}")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle("Reranker Training Data -- Length Distributions", fontsize=13)
plt.tight_layout()
plt.show()

## Cell 4 -- Configure Training Hyperparameters

All hyperparameters come from `config.yaml` via `src.constant`. Override any value here for experimentation. The defaults are tuned for a single A100 40 GB GPU.

In [ ]:
# ------------------------------------------------------------------
# Training configuration
# ------------------------------------------------------------------
TRAIN_CONFIG = dict(
    model_path  = constant.bge_reranker_model_path,
    train_file  = constant.reranker_train_path,
    eval_file   = constant.reranker_test_path,
    output_dir  = constant.bge_reranker_tuned_path,
    eval_dir    = str(Path(constant.BASE_DIR) / "outputs" / "reranker_eval"),
    epochs      = 3,
    batch_size  = 16,
    lr          = 2e-5,
    eval_steps  = 50,
    warmup_ratio = 0.1,
    max_length  = 512,
    fp16        = True,
)

print("Training configuration:")
for k, v in TRAIN_CONFIG.items():
    print(f"  {k:16s} = {v}")

## Cell 5 -- Launch Training

Call `train_reranker.train_reranker()` which handles the full training loop: data loading, optimizer/scheduler setup, periodic evaluation, model checkpointing, and TensorBoard logging. The function returns a `history` dict used for plotting.

In [ ]:
# ------------------------------------------------------------------
# Run fine-tuning  (takes ~10-30 min on a single A100 depending on data size)
# ------------------------------------------------------------------
history = train_reranker(**TRAIN_CONFIG)

print(f"\nTraining finished.")
print(f"  Total train steps recorded : {len(history['train_steps'])}")
print(f"  Total eval checkpoints     : {len(history['eval_steps'])}")

## Cell 6 -- Plot Training Curves (Loss, NDCG@10, MRR@10)

Visualize how training loss decreases and evaluation metrics improve over time. A divergence between train and eval loss would signal overfitting.

In [ ]:
# ------------------------------------------------------------------
# Three-panel training curves
# ------------------------------------------------------------------
eval_dir = TRAIN_CONFIG["eval_dir"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Loss
axes[0].plot(history["train_steps"], history["train_loss"],
             alpha=0.5, color="steelblue", linewidth=0.7, label="Train loss")
if history["eval_steps"]:
    axes[0].plot(history["eval_steps"], history["eval_loss"],
                 color="tomato", linewidth=2, marker="o", markersize=4, label="Eval loss")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("BCE Loss")
axes[0].set_title("Training & Evaluation Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Panel 2: NDCG@10
if history["ndcg10"]:
    axes[1].plot(history["eval_steps"], history["ndcg10"],
                 color="mediumseagreen", linewidth=2, marker="s", markersize=5)
    axes[1].set_xlabel("Step")
    axes[1].set_ylabel("NDCG@10")
    axes[1].set_title("NDCG@10 over Training")
    axes[1].grid(alpha=0.3)
    axes[1].set_ylim(0, 1.05)

# Panel 3: MRR@10
if history["mrr10"]:
    axes[2].plot(history["eval_steps"], history["mrr10"],
                 color="darkorange", linewidth=2, marker="^", markersize=5)
    axes[2].set_xlabel("Step")
    axes[2].set_ylabel("MRR@10")
    axes[2].set_title("MRR@10 over Training")
    axes[2].grid(alpha=0.3)
    axes[2].set_ylim(0, 1.05)

plt.suptitle("Reranker Fine-Tuning Curves", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Also save via the library function
plot_training_curves(history, eval_dir)
print(f"Charts saved to: {eval_dir}")

## Cell 7 -- Pre/Post Comparison Chart

Load both the original pre-trained reranker and the best fine-tuned checkpoint, evaluate them on the same held-out set, and display a grouped bar chart. This directly shows the gain from domain-specific fine-tuning.

In [ ]:
# ------------------------------------------------------------------
# Compare pre-trained vs fine-tuned reranker
# ------------------------------------------------------------------
finetuned_best = str(Path(TRAIN_CONFIG["output_dir"]) / "best")

compare_pre_post(
    model_path     = TRAIN_CONFIG["model_path"],
    finetuned_path = finetuned_best,
    eval_file      = TRAIN_CONFIG["eval_file"],
    output_dir     = eval_dir,
)

# Display the saved comparison chart inline
comparison_img = Path(eval_dir) / "comparison.png"
if comparison_img.exists():
    from IPython.display import Image, display
    display(Image(filename=str(comparison_img)))
else:
    print("Comparison chart not found -- ensure both models are available.")

## Summary

### Outputs Produced

| File | Description |
|------|-------------|
| `outputs/models/reranker_finetuned/best/` | Best checkpoint (highest NDCG@10) |
| `outputs/models/reranker_finetuned/final/` | Final checkpoint |
| `outputs/reranker_eval/reranker_training_curves.png` | Loss + metric curves |
| `outputs/reranker_eval/reranker_lr_schedule.png` | Learning rate schedule |
| `outputs/reranker_eval/comparison.png` | Pre- vs. post-fine-tuning bar chart |
| `outputs/reranker_eval/comparison.csv` | Numeric comparison results |

### Next Notebook

Proceed to **`10_train_llm_sft.ipynb`** to fine-tune the LLM (Llama 3.1 8B) with LoRA on immunology QA pairs.